<a href="https://colab.research.google.com/github/dheeraj-reddy8/T.Dheeraj-Reddy-AP24110011473-ML/blob/main/T_Dheeraj_Reddy_AP24110011473_Week11_NaiveBayes_Student_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 11 Lab: Naive Bayes Classifier

Name: T. Dheeraj Reddy

Registration No: AP24110011473

Section:C

**CSE303 Machine Learning | Unit III: Probability and Bayes Learning | Syllabus Exp 9 (naive Bayesian classifier and its accuracy)**

Pattern: **demo → practice → predict-then-run → deliberate error → reflection**.

| Part | Topic | Track |
|---|---|---|
| 1 | Probability and Bayes' theorem | core |
| 2 | Naive Bayes by hand (Play Tennis), and the zero-frequency error | core |
| 3 | Gaussian Naive Bayes in sklearn, log-probabilities, and scaling | core |
| 4 | Mini Project A: Naive Bayes on digits (continues Week 7/8/9) | image |
| 5 | Mini Project B: Naive Bayes on the six sentences (continues Week 7/8/9) | NLP |
| 6 | Reflection and capstone link | |

**Rule carried from Week 5:** split first; fit every scaler, PCA and tuned setting on **train only**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
np.set_printoptions(precision=4, suppress=True)

## Part 1: Probability and Bayes' Theorem
$$P(C\mid x)=\frac{P(x\mid C)\,P(C)}{P(x)}$$
**Posterior** = likelihood × prior / evidence. The evidence $P(x)=\sum_C P(x\mid C)P(C)$ is the same for every class, so to *compare* classes we only need the numerator.

### Predict-then-run
A disease affects 1% of people. A test is positive for 90% of sick people (sensitivity) and also for 5% of healthy people (false-positive rate). You test positive. Chance you are actually sick? Guess before running: **~90% / ~50% / ~15%**

In [2]:
prior, sens, fpr = 0.01, 0.90, 0.05           # P(sick), P(+|sick), P(+|healthy)
evidence = sens * prior + fpr * (1 - prior)    # P(+)
posterior = sens * prior / evidence            # P(sick | +)
print('P(positive)      =', round(evidence, 4))
print('P(sick | positive) =', round(posterior, 4))

# sanity check by simulation: 1,000,000 people
rng = np.random.default_rng(1); n = 1_000_000
sick = rng.random(n) < prior
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)
print('simulated          =', round(sick[pos].mean(), 4), f'({sick[pos].sum()} sick among {pos.sum()} positives)')

P(positive)      = 0.0585
P(sick | positive) = 0.1538
simulated          = 0.1545 (9035 sick among 58469 positives)


### Student Practice 1
An email filter knows: 40% of mail is spam; the word *free* appears in 50% of spam and 5% of non-spam.
1. Compute $P(\text{spam}\mid \text{free})$ by hand, then in code.
2. If the spam rate were only 5%, what would the posterior be? Why does it drop so much?
3. In the disease example, which term of Bayes' theorem do people usually ignore?

In [3]:
# Q1-Q2: your code here

# Q1: 40% spam rate
prior_1 = 0.40
lik_spam = 0.50
lik_ham = 0.05

evidence_1 = lik_spam * prior_1 + lik_ham * (1 - prior_1)
post_1 = (lik_spam * prior_1) / evidence_1
print(f"Q1 - P(spam | free) at 40% prior: {post_1:.4f}")

# Q2: 5% spam rate
prior_2 = 0.05
evidence_2 = lik_spam * prior_2 + lik_ham * (1 - prior_2)
post_2 = (lik_spam * prior_2) / evidence_2
print(f"Q2 - P(spam | free) at 5% prior : {post_2:.4f}")


Q1 - P(spam | free) at 40% prior: 0.8696
Q2 - P(spam | free) at 5% prior : 0.3448


## Part 2: Naive Bayes by Hand (Play Tennis)
Classic 14-day dataset (Mitchell, Ch. 6). Predict **Play** from four categorical features.

The **naive** assumption: features are independent *given the class*, so
$$P(x_1,\dots,x_d\mid C)=\prod_{j=1}^{d}P(x_j\mid C)\quad\Rightarrow\quad \hat C=\arg\max_C\; P(C)\prod_j P(x_j\mid C)$$

Steps: (1) class priors, (2) conditional probability tables from counts, (3) multiply, (4) normalise, (5) pick the largest.

| Day | Outlook | Temp | Humidity | Wind | Play |
|---|---|---|---|---|---|
| 1 | Sunny | Hot | High | Weak | No |
| 2 | Sunny | Hot | High | Strong | No |
| 3 | Overcast | Hot | High | Weak | Yes |
| 4 | Rain | Mild | High | Weak | Yes |
| 5 | Rain | Cool | Normal | Weak | Yes |
| 6 | Rain | Cool | Normal | Strong | No |
| 7 | Overcast | Cool | Normal | Strong | Yes |
| 8 | Sunny | Mild | High | Weak | No |
| 9 | Sunny | Cool | Normal | Weak | Yes |
| 10 | Rain | Mild | Normal | Weak | Yes |
| 11 | Sunny | Mild | Normal | Strong | Yes |
| 12 | Overcast | Mild | High | Strong | Yes |
| 13 | Overcast | Hot | Normal | Weak | Yes |
| 14 | Rain | Mild | High | Strong | No |

In [4]:
rows = [
 ('Sunny','Hot','High','Weak','No'), ('Sunny','Hot','High','Strong','No'), ('Overcast','Hot','High','Weak','Yes'),
 ('Rain','Mild','High','Weak','Yes'), ('Rain','Cool','Normal','Weak','Yes'), ('Rain','Cool','Normal','Strong','No'),
 ('Overcast','Cool','Normal','Strong','Yes'), ('Sunny','Mild','High','Weak','No'), ('Sunny','Cool','Normal','Weak','Yes'),
 ('Rain','Mild','Normal','Weak','Yes'), ('Sunny','Mild','Normal','Strong','Yes'), ('Overcast','Mild','High','Strong','Yes'),
 ('Overcast','Hot','Normal','Weak','Yes'), ('Rain','Mild','High','Strong','No')]
F = np.array([r[:-1] for r in rows]); y = np.array([r[-1] for r in rows])
features = ['Outlook', 'Temp', 'Humidity', 'Wind']
print('Priors: Yes =', (y == 'Yes').sum(), '/ 14,  No =', (y == 'No').sum(), '/ 14')
for j, name in enumerate(features):
    print(name)
    for v in sorted(set(F[:, j])):
        print(f'   {v:9} Yes {((F[:, j] == v) & (y == "Yes")).sum()}/9    No {((F[:, j] == v) & (y == "No")).sum()}/5')

Priors: Yes = 9 / 14,  No = 5 / 14
Outlook
   Overcast  Yes 4/9    No 0/5
   Rain      Yes 3/9    No 2/5
   Sunny     Yes 2/9    No 3/5
Temp
   Cool      Yes 3/9    No 1/5
   Hot       Yes 2/9    No 2/5
   Mild      Yes 4/9    No 2/5
Humidity
   High      Yes 3/9    No 4/5
   Normal    Yes 6/9    No 1/5
Wind
   Strong    Yes 3/9    No 3/5
   Weak      Yes 6/9    No 2/5


In [8]:
def nb_scores(q, alpha=0):
    """Unnormalised score P(C) * prod_j P(x_j|C) and the normalised posterior. alpha = Laplace smoothing."""
    out = {}
    for c in ['Yes', 'No']:
        Fc = F[y == c]
        p = float((y == c).mean())
        for j, v in enumerate(q):
            k = len(set(F[:, j]))                                  # how many values this feature can take
            p *= float((Fc[:, j] == v).sum() + alpha) / (len(Fc) + alpha * k)
        out[c] = p
    total = sum(out.values())
    return out, {c: out[c] / total for c in out}

### Predict-then-run
Today is **(Sunny, Cool, High, Strong)**. The prior favours *Yes* (9/14). Will Naive Bayes say Yes or No? Guess, then run.

In [9]:
q = ('Sunny', 'Cool', 'High', 'Strong')
scores, post = nb_scores(q)
print('scores   :', {c: round(v, 5) for c, v in scores.items()})
print('posterior:', {c: round(v, 4) for c, v in post.items()})
print('prediction:', max(post, key=post.get))

scores   : {'Yes': 0.00529, 'No': 0.02057}
posterior: {'Yes': 0.2046, 'No': 0.7954}
prediction: No


### Student Practice 2 (do on paper first, as in an exam)
For today = (Sunny, Cool, High, Strong):
1. Write $P(\text{Yes})\,P(\text{Sunny}|\text{Yes})\,P(\text{Cool}|\text{Yes})\,P(\text{High}|\text{Yes})\,P(\text{Strong}|\text{Yes})$ as a product of fractions, then evaluate it. Do the same for *No*.
2. Normalise: $P(\text{Yes}\mid x)$ = ___ %
3. Redo it with Laplace smoothing ($\alpha=1$). Does the prediction change? Does the probability?
4. **Accuracy on test days (Exp 9).** Using $\alpha=1$, classify these five unseen days and report the accuracy:

| Test day | Outlook | Temp | Humidity | Wind | True Play |
|---|---|---|---|---|---|
| T1 | Overcast | Mild | Normal | Weak | Yes |
| T2 | Rain | Hot | High | Strong | No |
| T3 | Sunny | Hot | Normal | Weak | Yes |
| T4 | Rain | Cool | High | Weak | Yes |
| T5 | Sunny | Mild | High | Strong | No |

5. Verify your function against sklearn's `CategoricalNB(alpha=1)` (hint: encode the strings with `OrdinalEncoder`).

In [10]:
tests = [(('Overcast','Mild','Normal','Weak'), 'Yes'), (('Rain','Hot','High','Strong'), 'No'),
         (('Sunny','Hot','Normal','Weak'), 'Yes'), (('Rain','Cool','High','Weak'), 'Yes'),
         (('Sunny','Mild','High','Strong'), 'No')]
# Q1-Q5: your code here

### Deliberate error: the zero-frequency problem
Today is **(Overcast, Hot, High, Strong)**. *Overcast* never occurs with *No* in the training data. Run the cell, then explain in one line why $\alpha=0$ is dangerous.

In [11]:
q2 = ('Overcast', 'Hot', 'High', 'Strong')
for a in [0, 1]:
    s, p = nb_scores(q2, alpha=a)
    print(f'alpha={a}: scores', {c: round(v, 5) for c, v in s.items()}, ' posterior', {c: round(float(v), 3) for c, v in p.items()})

alpha=0: scores {'Yes': 0.00705, 'No': 0.0}  posterior {'Yes': 1.0, 'No': 0.0}
alpha=1: scores {'Yes': 0.00885, 'No': 0.00683}  posterior {'Yes': 0.564, 'No': 0.436}


## Part 3: Gaussian Naive Bayes (continuous features)
For numeric features, model each feature *within each class* as a bell curve with that class's mean $\mu_{jc}$ and variance $\sigma^2_{jc}$:
$$P(x_j\mid C)=\frac{1}{\sqrt{2\pi\sigma^2_{jc}}}\exp\!\Big(-\frac{(x_j-\mu_{jc})^2}{2\sigma^2_{jc}}\Big)$$
sklearn stores these as `theta_` (means) and `var_` (variances). In practice we add **log-probabilities** instead of multiplying (see the deliberate error below).

In [12]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=1, stratify=yi)
gnb = GaussianNB().fit(Xi_tr, yi_tr)
print('class priors:', gnb.class_prior_)
print('class means (rows = setosa, versicolor, virginica):\n', gnb.theta_)
print('class variances:\n', gnb.var_)
print('test accuracy:', round(gnb.score(Xi_te, yi_te), 4))

class priors: [0.3333 0.3333 0.3333]
class means (rows = setosa, versicolor, virginica):
 [[5.0229 3.4343 1.4571 0.2429]
 [5.8714 2.78   4.2743 1.3229]
 [6.6771 2.9943 5.6371 2.0257]]
class variances:
 [[0.1275 0.132  0.0373 0.0076]
 [0.2015 0.0867 0.2042 0.0418]
 [0.3709 0.1125 0.3143 0.0831]]
test accuracy: 0.9778


### Student Practice 3
1. For the first test flower `x = Xi_te[0]`, compute the log-posterior of each class **yourself** from `gnb.class_prior_`, `gnb.theta_`, `gnb.var_` (Gaussian formula, summed over the 4 features). Which class wins? Does it match `gnb.predict`?
2. Repeat for every test row and check that your predictions equal `gnb.predict(Xi_te)`.
3. Print the confusion matrix. Which two species are confused?

In [13]:
# Q1-Q3: your code here

# Q1-Q3: your code here
iris_names = load_iris().target_names

# Helper function to compute unnormalized log-posterior: log P(C) + sum_j log P(x_j | C)
def manual_log_post(X_input):
    # Gaussian log-likelihood for each class c and feature j:
    # -0.5 * log(2 * pi * var) - ((x - mean)^2) / (2 * var)
    log_lik = -0.5 * np.sum(
        np.log(2 * np.pi * gnb.var_) + ((X_input[:, None, :] - gnb.theta_) ** 2) / gnb.var_,
        axis=2
    )
    return np.log(gnb.class_prior_) + log_lik

# 1. First test flower x = Xi_te[0]
log_post_0 = manual_log_post(Xi_te[:1])[0]
pred_0 = np.argmax(log_post_0)
print("Q1 - Unnormalised log-posteriors:", np.round(log_post_0, 4))
print(f"Q1 - Winning class: {pred_0} ({iris_names[pred_0]}), matches gnb.predict: {pred_0 == gnb.predict(Xi_te[:1])[0]}")

# 2. Repeat for every test row
my_preds = np.argmax(manual_log_post(Xi_te), axis=1)
sk_preds = gnb.predict(Xi_te)
print("Q2 - All test predictions match gnb.predict:", np.array_equal(my_preds, sk_preds))

# 3. Confusion matrix
cm = confusion_matrix(yi_te, my_preds)
print("Q3 - Confusion Matrix:\n", cm)

Q1 - Unnormalised log-posteriors: [-491.155   -17.3865   -2.944 ]
Q1 - Winning class: 2 (virginica), matches gnb.predict: True
Q2 - All test predictions match gnb.predict: True
Q3 - Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]


### Deliberate error: multiplying many small probabilities
Predict: what is the product of 200 probabilities, each equal to 0.01? Then run.

In [14]:
p = np.full(200, 0.01)
print('product     :', np.prod(p))
print('sum of logs :', np.sum(np.log(p)))

product     : 0.0
sum of logs : -921.0340371976183


### Predict-then-run
In Week 9, KNN needed scaling on Wine (unscaled PC1 was dominated by *proline*). Does standardising change Gaussian Naive Bayes? Guess: **helps a lot / no change / hurts**. Then run.

In [15]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=1, stratify=yw)
sw = StandardScaler().fit(Xw_tr)                       # fit on TRAIN only
Xw_tr_s, Xw_te_s = sw.transform(Xw_tr), sw.transform(Xw_te)
print('model            raw     scaled')
for name, mk in [('GaussianNB', lambda: GaussianNB()), ('KNN (K=3)', lambda: KNeighborsClassifier(3))]:
    a = mk().fit(Xw_tr, yw_tr).score(Xw_te, yw_te)
    b = mk().fit(Xw_tr_s, yw_tr).score(Xw_te_s, yw_te)
    print(f'{name:16} {a:.4f}  {b:.4f}')

model            raw     scaled
GaussianNB       0.9815  0.9815
KNN (K=3)        0.6667  0.9444


### Student Practice 4
1. Explain in one line why Gaussian NB does not care about feature units. (Look at $(x-\mu)^2/\sigma^2$: what happens to it if you change cm to mm?)
2. One split can be lucky. Report GaussianNB accuracy on the full Wine data with 5-fold cross-validation (mean and std). Why is it safe to skip the scaler here?

In [16]:
# Q2: your code here

scores = cross_val_score(GaussianNB(), Xw, yw, cv=5)
print(f"5-fold CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

5-fold CV accuracy: 0.9663 +/- 0.0211


# Mini Project A: Naive Bayes on Digits (Image Track — continuing from Weeks 7, 8 and 9)
Same split as before (`test_size=0.3, random_state=1, stratify`). Baselines you already have: **KNN (scaled, K=3) = 0.9796** and **Tree (entropy, full depth, unscaled) = 0.8815**. Question: how does Naive Bayes compare, and does Week 9's PCA help it?

In [17]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)
print('pixels with zero variance in train:', int((Xd_train.std(0) == 0).sum()), 'of 64')
print('GaussianNB test accuracy:', round(GaussianNB().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

pixels with zero variance in train: 3 of 64
GaussianNB test accuracy: 0.8315


### Predict-then-run
Before running the cell above, guess: will GaussianNB score **above both** baselines, **between** them, or **below both**? The 64 pixels are clearly *not* independent given the digit. Is that a problem for NB?

### Student Practice 5
1. Fit `MultinomialNB()` (treat pixel intensity as a count) and `BernoulliNB(binarize=8)` (ink / no ink) on the raw pixels. Report test accuracy for both.
2. GaussianNB has one knob, `var_smoothing`. For values `[1e-9, 1e-3, 1e-2, 1e-1, 1]`, use 5-fold cross-validation **on the training set only**, pick the best, then report its test accuracy **once**. (Why not pick the value with the best test accuracy?)
3. Using that best model, which two digits are confused most often?

In [19]:
# Q1-Q3: your code here

# Q1-Q3: your code here
# 1. MultinomialNB and BernoulliNB on raw pixels
mnb_acc = MultinomialNB().fit(Xd_train, yd_train).score(Xd_test, yd_test)
bnb_acc = BernoulliNB(binarize=8).fit(Xd_train, yd_train).score(Xd_test, yd_test)
print(f"Q1 - MultinomialNB test accuracy: {mnb_acc:.4f}")
print(f"Q1 - BernoulliNB(binarize=8) test accuracy: {bnb_acc:.4f}")

# 2. Tune var_smoothing using 5-fold CV on the training set only
smoothing_vals = [1e-9, 1e-3, 1e-2, 1e-1, 1]
cv_means = [
    cross_val_score(GaussianNB(var_smoothing=v), Xd_train, yd_train, cv=5).mean()
    for v in smoothing_vals
]
best_v = smoothing_vals[int(np.argmax(cv_means))]
best_gnb = GaussianNB(var_smoothing=best_v).fit(Xd_train, yd_train)
print(f"Q2 - Best var_smoothing (CV): {best_v}, test accuracy: {best_gnb.score(Xd_test, yd_test):.4f}")

# 3. Confusion matrix to find the most confused pair of digits
cm = confusion_matrix(yd_test, best_gnb.predict(Xd_test))
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)  # ignore correct predictions on the diagonal
i, j = np.unravel_index(np.argmax(cm_off), cm_off.shape)
print(f"Q3 - Most confused pair: true {i} predicted as {j} ({cm_off[i, j]} times)")
print("Confusion Matrix:\n", cm)

Q1 - MultinomialNB test accuracy: 0.8963
Q1 - BernoulliNB(binarize=8) test accuracy: 0.8963
Q2 - Best var_smoothing (CV): 0.01, test accuracy: 0.9296
Q3 - Most confused pair: true 1 predicted as 8 (6 times)
Confusion Matrix:
 [[52  0  0  0  2  0  0  0  0  0]
 [ 0 44  4  0  0  0  0  0  6  1]
 [ 0  0 51  1  0  0  0  0  1  0]
 [ 0  0  0 50  0  0  0  2  2  1]
 [ 0  0  0  0 50  0  0  3  1  0]
 [ 0  0  0  1  0 53  1  0  0  0]
 [ 0  0  0  0  2  0 52  0  0  0]
 [ 0  0  0  0  1  0  0 52  1  0]
 [ 0  4  0  0  0  1  0  1 46  0]
 [ 0  0  1  0  0  0  0  1  0 52]]


### Deliberate error: negative inputs
Since Week 9 we standardise pixels before PCA. Run the cell: what breaks, and why?

In [18]:
sc = StandardScaler().fit(Xd_train)                    # fit on TRAIN only
Xtr_s, Xte_s = sc.transform(Xd_train), sc.transform(Xd_test)
try:
    MultinomialNB().fit(Xtr_s, yd_train)
except ValueError as e:
    print('ValueError:', e)
print('GaussianNB on scaled pixels:', round(GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test), 4), ' (raw pixels gave 0.8315)')

ValueError: Negative values in data passed to MultinomialNB (input X).
GaussianNB on scaled pixels: 0.7759  (raw pixels gave 0.8315)


### Link to Week 9: does decorrelating the pixels help?
Naive Bayes assumes features are independent given the class. PCA scores are *uncorrelated by construction*. Question: does PCA before NB help?

### Student Practice 6
1. Compute the mean absolute off-diagonal correlation (a) of the non-constant raw pixels and (b) of the first 30 PCA scores of `Xtr_s`.
2. For k in `[10, 20, 30, 40, 64]` fit `PCA(k)` on `Xtr_s` (train only), then `GaussianNB` on the scores, and report test accuracy.
3. `PCA(64)` is only a rotation of the scaled pixels: no information is lost. So why does its accuracy differ from GaussianNB on scaled pixels (above)?

In [20]:
# Q1-Q3: your code here

# Q1-Q3: your code here
def mean_off_diag_corr(M):
    C = np.abs(np.corrcoef(M, rowvar=False))
    off = C[~np.eye(C.shape[0], dtype=bool)]
    return off.mean()

# 1a. Non-constant raw pixels (61 of 64 pixels have std > 0)
non_const_mask = Xd_train.std(axis=0) > 0
corr_raw = mean_off_diag_corr(Xd_train[:, non_const_mask])

# 1b. First 30 PCA scores of Xtr_s
pca_30 = PCA(n_components=30).fit(Xtr_s)
scores_30 = pca_30.transform(Xtr_s)
corr_pca = mean_off_diag_corr(scores_30)

print(f"Q1a - Mean |off-diag corr| (non-constant raw pixels): {corr_raw:.4f}")
print(f"Q1b - Mean |off-diag corr| (first 30 PCA scores)    : {corr_pca:.4f}\n")

# 2. Fit PCA(k) on Xtr_s (train only), then GaussianNB
for k in [10, 20, 30, 40, 64]:
    pca = PCA(n_components=k).fit(Xtr_s)
    Z_tr = pca.transform(Xtr_s)
    Z_te = pca.transform(Xte_s)
    acc = GaussianNB().fit(Z_tr, yd_train).score(Z_te, yd_test)
    print(f"Q2  - PCA(k={k:2d}) + GaussianNB test accuracy: {acc:.4f}")

Q1a - Mean |off-diag corr| (non-constant raw pixels): 0.1257
Q1b - Mean |off-diag corr| (first 30 PCA scores)    : 0.0000

Q2  - PCA(k=10) + GaussianNB test accuracy: 0.8296
Q2  - PCA(k=20) + GaussianNB test accuracy: 0.9148
Q2  - PCA(k=30) + GaussianNB test accuracy: 0.9259
Q2  - PCA(k=40) + GaussianNB test accuracy: 0.9222
Q2  - PCA(k=64) + GaussianNB test accuracy: 0.9185


# Mini Project B: Naive Bayes on the Six Sentences (NLP Track — continuing from Weeks 7, 8 and 9)
Same sentences and bag-of-words vectors. Naive Bayes is the classic text classifier (spam filters!). In Week 9, PCA failed to find the topics because *the* and *and* dominated the variance. Question: does NB do better?

In [21]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']
vocab = sorted(set(w for s in sentences for w in s.lower().split()))
def vec(s): return [s.lower().split().count(w) for w in vocab]       # words outside the vocabulary are ignored
X_text = np.array([vec(s) for s in sentences])
print('matrix shape (docs x vocab):', X_text.shape)
nb_text = MultinomialNB(alpha=1).fit(X_text, labels)
print('classes:', nb_text.classes_, ' priors:', np.exp(nb_text.class_log_prior_))

matrix shape (docs x vocab): (6, 29)
classes: ['cooking' 'sports' 'tech']  priors: [0.3333 0.3333 0.3333]


### Predict-then-run
Multinomial NB with smoothing uses $P(w\mid C)=\dfrac{\text{count}(w,C)+\alpha}{\text{total words in }C+\alpha\,|V|}$ with $|V|=29$.
(a) How many word tokens are in the *sports* class (two sentences)? (b) Which class will NB give to **'the new curry'**? Is it confident?

### Student Practice 7
1. Predict the five sentences below with `nb_text` (`alpha=1`) and print the class probabilities. Accuracy?
   `'the team scored a century'` (sports), `'boil the pasta with salt'` (cooking), `'the phone battery lasts a day'` (tech), `'the match today'` (sports), `'the new curry'` (cooking)
2. By hand: for `'the team scored a century'` compute the three class scores from the count formula above and normalise. Compare with `predict_proba`.
3. Print the 5 words with the highest $P(w\mid C)$ in each class. Is *the* near the top everywhere? Why does that **not** hurt Naive Bayes, when it did hurt PCA?
4. Why is the probability for **'the new curry'** so spread out?

In [22]:
# Q1-Q4: your code here

# Q1-Q4: your code here
test_sents = [
    ('the team scored a century', 'sports'),
    ('boil the pasta with salt', 'cooking'),
    ('the phone battery lasts a day', 'tech'),
    ('the match today', 'sports'),
    ('the new curry', 'cooking')
]

X_test_s = np.array([vec(s) for s, _ in test_sents])
y_test_s = [lbl for _, lbl in test_sents]

preds = nb_text.predict(X_test_s)
probs = nb_text.predict_proba(X_test_s)

print("Classes:", nb_text.classes_)
for (sent, true_lbl), pred, prob in zip(test_sents, preds, probs):
    print(f"'{sent}' -> Pred: {pred:7s} (True: {true_lbl:7s}) | Probs: {np.round(prob, 4)}")

acc = np.mean(preds == y_test_s)
print(f"\nQ1 - Accuracy: {acc:.2f} ({int(acc * len(y_test_s))}/{len(y_test_s)})\n")

# 3. Top 5 words with highest P(w | C) in each class
feature_probs = np.exp(nb_text.feature_log_prob_)
for idx, c in enumerate(nb_text.classes_):
    top5_idx = np.argsort(feature_probs[idx])[::-1][:5]
    top5_words = [(vocab[i], round(float(feature_probs[idx][i]), 4)) for i in top5_idx]
    print(f"Q3 - Top 5 in {c:7s}: {top5_words}")

Classes: ['cooking' 'sports' 'tech']
'the team scored a century' -> Pred: sports  (True: sports ) | Probs: [0.0487 0.8783 0.073 ]
'boil the pasta with salt' -> Pred: cooking (True: cooking) | Probs: [0.8121 0.1118 0.0761]
'the phone battery lasts a day' -> Pred: tech    (True: tech   ) | Probs: [0.0366 0.0846 0.8788]
'the match today' -> Pred: sports  (True: sports ) | Probs: [0.1653 0.7107 0.124 ]
'the new curry' -> Pred: cooking (True: cooking) | Probs: [0.4372 0.235  0.3279]

Q1 - Accuracy: 1.00 (5/5)

Q3 - Top 5 in cooking: [('the', 0.0952), ('and', 0.0714), ('simmer', 0.0476), ('pasta', 0.0476), ('onions', 0.0476)]
Q3 - Top 5 in sports : [('the', 0.0976), ('won', 0.0488), ('today', 0.0488), ('team', 0.0488), ('scored', 0.0488)]
Q3 - Top 5 in tech   : [('the', 0.0714), ('processor', 0.0476), ('new', 0.0476), ('phone', 0.0476), ('lasts', 0.0476)]


### Deliberate error: almost no smoothing
`'the cricket phone'` mixes a sports word with a tech word. Run the cell and compare how *confident* each model is.

In [23]:
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a).fit(X_text, labels)
    print(f'alpha={a:<6}', dict(zip(m.classes_.tolist(), m.predict_proba([vec('the cricket phone')])[0].round(3).tolist())))

alpha=1e-10  {'cooking': 0.0, 'sports': 0.656, 'tech': 0.344}
alpha=1.0    {'cooking': 0.215, 'sports': 0.462, 'tech': 0.323}


## Part 6: Reflection (write 2–3 lines each)
1. In your own words: what is the "naive" assumption, and what does it make easy?
2. What is the zero-frequency problem? Show it with the Play Tennis example and say how Laplace smoothing fixes it.
3. Why do real implementations add log-probabilities instead of multiplying probabilities?
4. Wine: scaling changed KNN but not Gaussian NB. What does that tell you about how each model uses the features?
5. Digits: PCA helped NB but hurt the tree (Week 9). Explain using the independence assumption.
6. Naive Bayes is fast and needs little data, but its probabilities are often overconfident. When would you still choose it?
7. **Capstone link (Milestone, Week 10):** will Naive Bayes belong in your pipeline, at least as a baseline? If yes, add its accuracy to your comparison table in your repo's `notebooks/` folder; if no, justify.

In [ ]:
Here are concise 2–3 line answers for **Part 6: Reflection**:

1. **The "naive" assumption & what it simplifies:**
The naive assumption is that all features are conditionally independent of one another given the class label. This makes computation and training dramatically easier because instead of estimating a complex joint distribution across all feature combinations, we only need to estimate each feature's 1D distribution separately and multiply them together.
2. **The zero-frequency problem & Laplace smoothing:**
If a feature value never appears with a given class in the training set, its conditional probability is $0$, which wipes out the entire product for that class regardless of strong evidence from other features. In Play Tennis, `Outlook = Overcast` never appears with `Play = No` ($0/5$), forcing $P(\text{No} \mid x) = 0$; Laplace smoothing ($\alpha = 1$) adds a small pseudocount to every numerator and $\vert{}V\vert{}$ to the denominator so no probability is ever strictly zero.
3. **Why real implementations add log-probabilities:**
Multiplying many small probabilities (e.g., $0.01^{200}$) quickly causes floating-point **underflow**, rounding the product down to `0.0` so classes can no longer be distinguished. Converting products into sums of log-probabilities ($\log(ab) = \log a + \log b$) keeps numbers in a safe, well-scaled negative range while preserving the exact class ranking.
4. **Wine: scaling in KNN vs. Gaussian NB:**
KNN relies on Euclidean distance across all features combined, so large-scale features like *proline* dominate the distance metric unless standardized. Gaussian NB evaluates each feature independently within its own distribution using z-scores $(x - \mu)^2 / \sigma^2$, making it naturally scale-invariant.
5. **Digits: why PCA helped NB but hurt Decision Trees:**
Raw image pixels are strongly correlated with their neighbors, violating Naive Bayes' conditional independence assumption; PCA decorrelates the features into orthogonal axes, making NB's diagonal-covariance assumption fit much better. Decision Trees, however, rely on axis-aligned splits on individual interpretable pixels, and rotating the space into dense linear combinations forces trees to use jagged staircase approximations.
6. **When to still choose Naive Bayes despite overconfident probabilities:**
Naive Bayes is ideal as a fast, low-memory baseline, on small training datasets, or in high-dimensional sparse text tasks (like spam filtering and document classification). Because classification only depends on which class has the *highest* score rather than calibrated probability values, overconfidence does not hurt accuracy when you only need the predicted label.
7. **Capstone link (Milestone, Week 10):**
*(Adapt to your project's data type)* **Yes**—Naive Bayes belongs in the pipeline as a fast, leak-free baseline model because it trains in milliseconds and establishes a clear benchmark before tuning heavier models (using `MultinomialNB` / `BernoulliNB` if your capstone uses text/count features, or `GaussianNB` if it uses continuous tabular features). *(If your capstone strictly uses image pixels or raw audio/time-series where spatial/temporal structure is critical, note that it serves only as a simple tabular baseline before moving to feature extraction or specialized models.)*

*Submit this notebook to GitHub under `labs/week10_naive_bayes/`.*